# 🔧 Notebook 1 — Ingeniería de Índices LEPI
**Habilidades demostradas:** Feature Engineering · Índices compuestos · EDA profesional  

---
**Caso de negocio:** La app LEPI necesita convertir 12 variables de encuesta
en 4 índices interpretables (ILD, IRPD, IIP, ILEPI) para evaluar docentes.
Este notebook documenta el diseño de esas métricas con validación estadística.


In [ ]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')
from data_generator import generar_docentes, generar_impacto, resumen_institucional
df = generar_docentes(300)
df_impacto = generar_impacto(15)
print(f"✅ Datos cargados — {len(df)} docentes | {len(df_impacto)} instituciones")
df.head()


## 1. Diseño de los Índices LEPI

In [ ]:
# ─── Pesos del modelo LEPI ─────────────────────────────────
PESOS = {"ILD": 0.35, "IRPD": 0.35, "IIP": 0.30}
DIMS  = {
    "ILD":  ["lid_vision","lid_comunicacion","lid_colaboracion","lid_motivacion","lid_resolucion"],
    "IRPD": ["prax_ver","prax_juzgar","prax_actuar","prax_devolver"],
    "IIP":  ["inn_tecnologia","inn_metodologia","inn_evaluacion"],
}
print("Estructura de los índices LEPI:")
for idx, dims in DIMS.items():
    print(f"  {idx} ({PESOS[idx]*100:.0f}%): {', '.join(dims)}")
print(f"\n  ILEPI = ILD×{PESOS['ILD']} + IRPD×{PESOS['IRPD']} + IIP×{PESOS['IIP']}")


## 2. EDA — Exploración de Variables

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('EDA — Índices LEPI | Portafolio Analista de Datos', fontsize=14, fontweight='bold')

# Distribuciones de índices
for ax, (col, color) in zip(axes[0],
    [("ILD","#2E75B6"),("IRPD","#27AE60"),("IIP","#E74C3C")]):
    ax.hist(df[col], bins=25, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(df[col].mean(), color='black', linestyle='--',
               label=f"μ={df[col].mean():.2f}")
    ax.axvspan(1, 3.0, alpha=0.06, color='red')
    ax.axvspan(3.0, 3.5, alpha=0.06, color='orange')
    ax.axvspan(3.5, 4.0, alpha=0.06, color='yellow')
    ax.axvspan(4.0, 5.0, alpha=0.06, color='green')
    ax.set_title(f'Distribución {col}'); ax.legend(fontsize=9)

# Heatmap correlaciones
corr_cols = ['ILD','IRPD','IIP','ILEPI','satisfaccion','anos_exp','horas_formacion']
corr = df[corr_cols].corr()
sns.heatmap(corr, ax=axes[1,0], annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, linewidths=0.5, vmin=-1, vmax=1)
axes[1,0].set_title('Mapa de Correlaciones')

# ILEPI por nivel de formación
form_ilepi = df.groupby('formacion')['ILEPI'].mean().sort_values()
axes[1,1].barh(form_ilepi.index, form_ilepi.values,
               color=plt.cm.Blues(np.linspace(0.4,1.0,4)))
axes[1,1].axvline(3.5, color='red', linestyle='--', label='Meta 3.5')
axes[1,1].set_title('ILEPI Promedio por Formación'); axes[1,1].legend()

# ILEPI vs Años de experiencia
axes[1,2].scatter(df['anos_exp'], df['ILEPI'],
                  c=df['ILD'], cmap='Blues', alpha=0.5, s=25)
axes[1,2].set_xlabel('Años de experiencia'); axes[1,2].set_ylabel('ILEPI')
axes[1,2].set_title('ILEPI vs Experiencia (color=ILD)')

plt.tight_layout(); plt.show()


## 3. Validación Psicométrica — Alpha de Cronbach

In [ ]:
def cronbach_alpha(df_items: pd.DataFrame) -> float:
    """Calcula la consistencia interna del índice."""
    k = df_items.shape[1]
    var_items = df_items.var(axis=0, ddof=1).sum()
    var_total = df_items.sum(axis=1).var(ddof=1)
    return (k / (k-1)) * (1 - var_items/var_total) if var_total > 0 else 0

print("Validación de Consistencia Interna (Alpha de Cronbach):")
print("─" * 52)
for idx, dims in DIMS.items():
    disponibles = [d for d in dims if d in df.columns]
    alpha = cronbach_alpha(df[disponibles])
    nivel = "Excelente" if alpha>0.9 else "Bueno" if alpha>0.8 else "Aceptable" if alpha>0.7 else "Revisar"
    estado = "✅" if alpha > 0.7 else "⚠️"
    print(f"  {idx:<6}: α = {alpha:.4f}  → {nivel} {estado}")
print("─" * 52)
print("  Referencia: α > 0.70 es aceptable para escalas de actitud")


## 4. Análisis por Componente LEPI

In [ ]:
print("Estadísticas por Índice LEPI:")
print(df[['ILD','IRPD','IIP','ILEPI']].agg(
    ['mean','std','min','median','max']).round(3))
print(f"\nDistribución de Niveles LEPI:")
print(df['nivel_lepi'].value_counts())
print(f"\n% docentes por debajo de 3.0 (zona de riesgo):")
for col in ['ILD','IRPD','IIP','ILEPI']:
    pct = (df[col]<3.0).mean()*100
    print(f"  {col}: {pct:.1f}%  {'⚠️' if pct>20 else '✅'}")


## ✅ Conclusiones Técnicas

### Habilidades demostradas
- **Feature Engineering**: Diseño de 4 índices ponderados desde 12 variables originales
- **Validación estadística**: Alpha de Cronbach para consistencia interna
- **EDA profesional**: Correlaciones, distribuciones, análisis por segmento

### Hallazgos clave para la app LEPI
- El componente **IIP (Innovación)** es el más bajo — área prioritaria de intervención
- El **IRPD** muestra la mayor varianza — heterogeneidad en práctica reflexiva
- Correlación **formación académica → ILEPI** confirma valor del desarrollo profesional
